# FlashInspector AI – Train on Kaggle (GPU)

Train the fire safety YOLO model on Kaggle's free GPU.

**Before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 x2).
2. Add your [Roboflow API key](https://app.roboflow.com/settings/api) as a **Secret**: Notebook setup → Add-ons → Secrets → add `ROBOFLOW_API_KEY`, or paste it when prompted below.
3. This notebook clones the repo from GitHub; set `REPO_URL` in the next cell if using a fork.

## 1. Check GPU

In [ ]:
!nvidia-smi 2>/dev/null || echo "No GPU. Enable: Settings → Accelerator → GPU (P100 or T4)"

## 2. Clone repo and go to project

In [ ]:
REPO_URL = "https://github.com/patrisiyarum/fire.git"

!git clone --depth 1 $REPO_URL /kaggle/working/fire
%cd /kaggle/working/fire/flashinspector-ai
import os
print("Project dir:", os.getcwd())

## 3. Install dependencies

You may see dependency conflict warnings; **ignore them** — training will still work.

In [ ]:
!pip install -q --upgrade-strategy only-if-needed ultralytics roboflow opencv-python python-dotenv pyyaml tqdm

## 4. Roboflow API key

**Option A (easiest):** Replace `paste_your_key_here` below with your [Roboflow API key](https://app.roboflow.com/settings/api), then run the cell. **Do not commit or share the notebook with the key in it.**

**Option B:** Use Kaggle Secrets (Add-ons → Secrets, add `ROBOFLOW_API_KEY`); the cell will use it if set.

In [ ]:
import os

# Paste your key here (replace the string) if Secrets aren't working:
ROBOFLOW_API_KEY = "paste_your_key_here"

if os.environ.get("ROBOFLOW_API_KEY"):
    print("Using ROBOFLOW_API_KEY from Secrets.")
elif ROBOFLOW_API_KEY and ROBOFLOW_API_KEY != "paste_your_key_here":
    os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
    print("Using ROBOFLOW_API_KEY from this cell.")
else:
    raise RuntimeError(
        "Set your key: replace paste_your_key_here above with your Roboflow API key, or add ROBOFLOW_API_KEY in Add-ons → Secrets."
    )

## 5. Download all datasets

In [ ]:
!python download_datasets.py

## 6. Merge datasets (multi-class)

In [ ]:
!python prepare_dataset.py

## 7. Train on Kaggle GPU

Trains on the merged dataset (all classes). Output goes to `runs/fire_safety/weights/best.pt`. Colab-friendly: 50 epochs, batch 8.

In [ ]:
!python train.py --epochs 50 --batch 8 --model yolov8n.pt

## 8. Save model to Kaggle Output

Copy `best.pt` to `/kaggle/working/` so it appears in **Output** and you can download it after the run.

In [ ]:
import shutil
from pathlib import Path

src = Path("runs/fire_safety/weights/best.pt")
if src.exists():
    dest = Path("/kaggle/working/best.pt")
    shutil.copy(src, dest)
    print(f"Saved to {dest} — download from the Output tab when the run finishes.")
else:
    print("best.pt not found. Check training output above.")
    !ls -la runs/fire_safety/weights/ 2>/dev/null || true

## 9. Test the model in Kaggle

Run inference on an image or video and see the result in the notebook.

- **Same run:** Model is at `runs/fire_safety/weights/best.pt` (or `/kaggle/working/best.pt` if you ran Section 8).
- **New run (model from Output):** Add your previous run’s Output as a dataset, then set `MODEL_PATH` to e.g. `/kaggle/input/your-output-name/best.pt`.

**Input:** Use a sample from the downloaded dataset, or add an image/video via **Add Data → Upload** and set `TEST_IMAGE` to its path (e.g. `/kaggle/input/your-upload/image.jpg`).

In [ ]:
# Model path: local run first, then Kaggle working, then input (if you added Output as dataset)
from pathlib import Path
import os

PROJECT_DIR = Path("/kaggle/working/fire/flashinspector-ai")
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)
print("Project dir:", PROJECT_DIR)

MODEL_PATH = Path("runs/fire_safety/weights/best.pt")
if not MODEL_PATH.exists():
    MODEL_PATH = Path("/kaggle/working/best.pt")
if not MODEL_PATH.exists():
    # If you added a previous run's Output as input dataset:
    import glob
    candidates = glob.glob("/kaggle/input/*/best.pt")
    MODEL_PATH = Path(candidates[0]) if candidates else None
if MODEL_PATH is None or not Path(MODEL_PATH).exists():
    raise FileNotFoundError("No best.pt found. Train first (Section 7) and optionally save (Section 8), or add Output as input and set path.")
print("Using model:", MODEL_PATH)

In [ ]:
# Test image/video: use a dataset sample, or set path if you added data via "Add Data → Upload"
TEST_IMAGE = ""  # e.g. "/kaggle/input/my-images/fire_test.jpg" or leave empty to use dataset sample

if not TEST_IMAGE:
    sample_dir = PROJECT_DIR / "fire_safety_datasets" / "fire_extinguisher" / "valid" / "images"
    if sample_dir.exists():
        samples = list(sample_dir.glob("*.*"))[:1]
        TEST_IMAGE = str(samples[0]) if samples else ""
if not TEST_IMAGE:
    raise SystemExit("Set TEST_IMAGE to an image path (e.g. from Add Data), or run Section 5 so a dataset sample exists.")

test_path = Path(TEST_IMAGE)
if not test_path.is_absolute():
    test_path = (PROJECT_DIR / test_path).resolve()
if not test_path.exists():
    raise FileNotFoundError(f"Not found: {test_path}")
TEST_IMAGE = str(test_path)
print("Input:", TEST_IMAGE)

In [ ]:
# Run inference (image = direct; video = test_model.py)
from ultralytics import YOLO
import cv2
import subprocess

out_dir = PROJECT_DIR / "inference_results"
out_dir.mkdir(exist_ok=True)
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

if test_path.suffix.lower() in VIDEO_EXTS:
    script_path = PROJECT_DIR / "fire_safety_datasets" / "test_model.py"
    r = subprocess.run(
        ["python", str(script_path), TEST_IMAGE, "--model", str(MODEL_PATH), "--conf", "0.25", "--frame-skip", "5"],
        cwd=str(PROJECT_DIR), capture_output=True, text=True
    )
    print(r.stdout)
    if r.stderr:
        print("STDERR:", r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"test_model.py failed (code {r.returncode})")
else:
    model = YOLO(str(MODEL_PATH))
    results = model(TEST_IMAGE, conf=0.25, verbose=True)[0]
    out_path = out_dir / f"result_{test_path.name}"
    cv2.imwrite(str(out_path), results.plot())
    print("Saved to:", out_path)

In [ ]:
# Show result in notebook (image) or path (video)
from IPython.display import Image, display

name = test_path.name
stem = test_path.stem
result_path = out_dir / f"result_{name}"
if not result_path.exists():
    result_path = out_dir / f"result_{stem}.mp4"
if result_path.exists():
    if result_path.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"):
        display(Image(str(result_path), width=600))
    else:
        print("Annotated video saved:", result_path)
    print("Output:", result_path)
else:
    print("Result not found. Check inference cell above.")
    import os
    print(os.listdir(out_dir) if out_dir.exists() else "inference_results/ missing")